In [23]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/bipasharouthbr/inidan-acts/Consumer_Protection_Act_2019.pdf
/kaggle/input/datasets/bipasharouthbr/inidan-acts/IT_Act_2000.pdf
/kaggle/input/datasets/bipasharouthbr/inidan-acts/RTI-Act_2005.pdf


In [25]:
!pip -q install pymupdf rank_bm25 sentence-transformers faiss-cpu groq gradio pandas

In [26]:
import glob, os

paths = glob.glob("/kaggle/input/**/*.pdf", recursive=True)
PDFS = {p: os.path.splitext(os.path.basename(p))[0].replace("_", " ").replace("-", " ") for p in paths}

for p, n in PDFS.items():
    print(n, "->", p)

Consumer Protection Act 2019 -> /kaggle/input/datasets/bipasharouthbr/inidan-acts/Consumer_Protection_Act_2019.pdf
IT Act 2000 -> /kaggle/input/datasets/bipasharouthbr/inidan-acts/IT_Act_2000.pdf
RTI Act 2005 -> /kaggle/input/datasets/bipasharouthbr/inidan-acts/RTI-Act_2005.pdf


chunking

In [28]:
import fitz, re, json

def load_chunks(pdf_path, act, max_chars=1000, overlap=150):
    doc = fitz.open(pdf_path)
    out = []
    for pno, page in enumerate(doc, 1):
        text = re.sub(r"[ \t]+", " ", page.get_text())
        parts = re.split(r"\n(?=\d{1,3}[A-Z]?\.\s)", text)   # split at section numbers
        for p in parts:
            p = p.strip()
            if len(p) < 80:
                continue
            m = re.match(r"(\d{1,3}[A-Z]?)\.", p)
            sec = m.group(1) if m else None
            start = 0
            while start < len(p):
                out.append({"text": p[start:start + max_chars],
                            "act": act, "page": pno, "section": sec})
                if start + max_chars >= len(p):
                    break
                start += max_chars - overlap
    return out

chunks = []
for path, act in PDFS.items():
    c = load_chunks(path, act)
    print(act, "->", len(c), "chunks")
    chunks += c

print("Total:", len(chunks))
print(chunks[50])

Consumer Protection Act 2019 -> 210 chunks
IT Act 2000 -> 233 chunks
RTI Act 2005 -> 111 chunks
Total: 554
{'text': 'SEC. 1]\nTHE GAZETTE OF INDIA EXTRAORDINARY\n13\n(5) The inquiries or the investigations made by the Director- General shall be submitted\nto the Central Authority in such form, in such manner and within such time, as may be\nspecified by regulations.', 'act': 'Consumer Protection Act 2019', 'page': 13, 'section': None}


dense and BM25

In [30]:
import numpy as np, faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

emb = SentenceTransformer("BAAI/bge-small-en-v1.5")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

texts = [c["text"] for c in chunks]
vecs = emb.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=True)

index = faiss.IndexFlatIP(vecs.shape[1])
index.add(vecs)

tok = lambda s: re.findall(r"\w+", s.lower())
bm25 = BM25Okapi([tok(t) for t in texts])
print("Indexes ready:", len(texts))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Indexes ready: 554


Retrieval(BM25,dense,hybrid,hybrid+rerank)

In [33]:
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def retrieve(query, k=20, final_k=5, mode="hybrid_rerank"):
    qv = emb.encode([QUERY_PREFIX + query], normalize_embeddings=True)
    _, d = index.search(qv, k)
    dense_ids = d[0].tolist()
    sparse_ids = np.argsort(bm25.get_scores(tok(query)))[::-1][:k].tolist()

    if mode == "dense": return dense_ids[:final_k]
    if mode == "bm25":  return sparse_ids[:final_k]

    scores = {}
    for ranking in (dense_ids, sparse_ids):            # Reciprocal Rank Fusion
        for r, i in enumerate(ranking):
            scores[i] = scores.get(i, 0) + 1 / (60 + r)
    fused = sorted(scores, key=scores.get, reverse=True)[:k]
    if mode == "hybrid": return fused[:final_k]

    rr = reranker.predict([(query, texts[i]) for i in fused])
    return [fused[i] for i in np.argsort(rr)[::-1][:final_k]]

# quick test
for i in retrieve("How many days does a PIO have to reply to an RTI application?"):
    print(chunks[i]["act"], "| p.", chunks[i]["page"], "| sec", chunks[i]["section"])

Consumer Protection Act 2019 | p. 27 | sec 60
Consumer Protection Act 2019 | p. 22 | sec 40
RTI Act 2005 | p. 10 | sec None
Consumer Protection Act 2019 | p. 24 | sec 50
RTI Act 2005 | p. 19 | sec 19


In [35]:
from groq import Groq
from kaggle_secrets import UserSecretsClient

client = Groq(api_key=UserSecretsClient().get_secret("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-120b"

In [36]:
models = [m.id for m in client.models.list().data]
for m in sorted(models):
    print(m)

allam-2-7b
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
qwen/qwen3.8-27b
whisper-large-v3
whisper-large-v3-turbo


In [40]:
from groq import Groq
from kaggle_secrets import UserSecretsClient

client = Groq(api_key=UserSecretsClient().get_secret("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-120b"

def llm_call(system, user, temperature=0):
    r = client.chat.completions.create(
        model=MODEL, temperature=temperature, max_tokens=1500,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}])
    return r.choices[0].message.content

SYSTEM = """You answer questions about Indian law using ONLY the provided context.
Cite every claim as [Act, Section X, p.Y]. If the context does not contain the answer,
reply exactly: "I could not find this in the provided documents."
Be concise. End with: "This is general information, not legal advice." """

def answer(query, mode="hybrid_rerank"):
    ids = retrieve(query, mode=mode)
    ctx = "\n\n".join(
        f"[{chunks[i]['act']}, Section {chunks[i]['section']}, p.{chunks[i]['page']}]\n{chunks[i]['text']}"
        for i in ids)
    return llm_call(SYSTEM, f"Context:\n{ctx}\n\nQuestion: {query}"), ids

ans, ids = answer("What is the penalty for not providing information under the RTI Act?")
print(ans)

The RTI Act provides that if a Public Information Officer refuses to receive an application or fails to furnish information within the prescribed time, a penalty of **₹250 per day** may be imposed until the application is received or the information is furnished, **capped at a maximum of ₹25,000**【RTI Act 2005, Section 20, p.20】.  

This is general information, not legal advice.


In [42]:
ans, ids = answer("What is the penalty for not providing information under the RTI Act?")
print(ans)

Under the Right to Information Act, if the Public Information Officer refuses to receive an application or fails to furnish the information within the prescribed time, a penalty of ₹250 per day is imposed until the application is received or the information is provided, with the total penalty capped at ₹25,000 [RTI Act 2005, Section 20, p.20].  

This is general information, not legal advice.


building the evaluation set

In [41]:
import random, time, json

random.seed(42)
cands = [i for i, c in enumerate(chunks) if len(c["text"]) > 500]
sample = random.sample(cands, 70)

GEN = """Write ONE specific question a citizen might ask that is answerable ONLY from the passage below.
Do not say "the passage" or "the text". Return just the question.

Passage:
{p}"""

eval_set = []
for n, i in enumerate(sample):
    try:
        q = llm_call("You write evaluation questions.", GEN.format(p=chunks[i]["text"]), temperature=0.3).strip()
    except Exception as e:
        print("skip", n, e)
        time.sleep(5)
        continue
    eval_set.append({"question": q, "act": chunks[i]["act"], "page": chunks[i]["page"], "chunk_id": i})
    time.sleep(0.5)

json.dump(eval_set, open("/kaggle/working/eval_raw.json", "w"), indent=1)
for n, e in enumerate(eval_set):
    print(n, "|", e["act"], "p.", e["page"], "|", e["question"])

0 | Consumer Protection Act 2019 p. 18 | What is the definition of a “recognised consumer association” in the passage?
1 | Consumer Protection Act 2019 p. 5 | What four categories of harm are included in the definition of “harm” for product liability?
2 | IT Act 2000 p. 7 | What is the definition of “electronic signature” as provided in the passage?
3 | Consumer Protection Act 2019 p. 38 | What does clause (p) specify about the consumer mediation cell?
4 | Consumer Protection Act 2019 p. 35 | After how many years does a second or subsequent offence become deemed a first offence under this provision?
5 | Consumer Protection Act 2019 p. 22 | What is the time limit for a member to give his opinion on points referred to him in a proceeding conducted by the President and a member?
6 | Consumer Protection Act 2019 p. 16 | What is the minimum number of members, aside from the President, that a District Consumer Disputes Redressal Commission must have?
7 | RTI Act 2005 p. 15 | If a person appo

In [43]:
bad = []   # put your indices here, e.g. [3, 11, 25, 40]
eval_set = [e for n, e in enumerate(eval_set) if n not in bad][:50]
json.dump(eval_set, open("/kaggle/working/eval_final.json", "w"), indent=1)
print(len(eval_set), "questions kept")

50 questions kept


In [44]:
import pandas as pd

def evaluate(eval_set, mode, k=5):
    hits, rr = 0, 0
    for q in eval_set:
        ids = retrieve(q["question"], final_k=k, mode=mode)
        rank = next((r for r, i in enumerate(ids)
                     if chunks[i]["act"] == q["act"] and abs(chunks[i]["page"] - q["page"]) <= 1), None)
        if rank is not None:
            hits += 1
            rr += 1 / (rank + 1)
    return hits / len(eval_set), rr / len(eval_set)

rows = []
for mode in ["bm25", "dense", "hybrid", "hybrid_rerank"]:
    h, m = evaluate(eval_set, mode)
    rows.append({"method": mode, "Hit@5": round(h, 3), "MRR": round(m, 3)})

results = pd.DataFrame(rows)
print(results.to_markdown(index=False))
results.to_csv("/kaggle/working/results.csv", index=False)

| method        |   Hit@5 |   MRR |
|:--------------|--------:|------:|
| bm25          |    0.96 | 0.833 |
| dense         |    0.94 | 0.819 |
| hybrid        |    0.98 | 0.833 |
| hybrid_rerank |    1    | 0.932 |


ablation table

In [45]:
import pandas as pd

def evaluate(eval_set, mode, k=5):
    hits, rr = 0, 0
    for q in eval_set:
        ids = retrieve(q["question"], final_k=k, mode=mode)
        rank = next((r for r, i in enumerate(ids)
                     if chunks[i]["act"] == q["act"] and abs(chunks[i]["page"] - q["page"]) <= 1), None)
        if rank is not None:
            hits += 1
            rr += 1 / (rank + 1)
    return hits / len(eval_set), rr / len(eval_set)

rows = []
for mode in ["bm25", "dense", "hybrid", "hybrid_rerank"]:
    h, m = evaluate(eval_set, mode)
    rows.append({"method": mode, "Hit@5": round(h, 3), "MRR": round(m, 3)})

results = pd.DataFrame(rows)
print(results.to_markdown(index=False))
results.to_csv("/kaggle/working/results.csv", index=False)

| method        |   Hit@5 |   MRR |
|:--------------|--------:|------:|
| bm25          |    0.96 | 0.833 |
| dense         |    0.94 | 0.819 |
| hybrid        |    0.98 | 0.833 |
| hybrid_rerank |    1    | 0.932 |


In [46]:
hard_set = [
    {"question": "My employer-style example: a shop refused to refund a defective phone, where can I complain?",
     "act": "Consumer Protection Act 2019", "page": 0},   # <- set the real page
    # add 10-15 of these, each with the correct act name and page
]

rows = []
for mode in ["bm25", "dense", "hybrid", "hybrid_rerank"]:
    h, m = evaluate(hard_set, mode)
    rows.append({"method": mode, "Hit@5": round(h, 3), "MRR": round(m, 3)})
print(pd.DataFrame(rows))

          method  Hit@5  MRR
0           bm25    0.0  0.0
1          dense    0.0  0.0
2         hybrid    0.0  0.0
3  hybrid_rerank    0.0  0.0


In [47]:
tests = [
    "How long does an authority have to respond to an information request?",
    "What can I do if a product I bought is defective?",
    "What is the capital of France?",
]

for q in tests:
    ans, ids = answer(q)
    print("Q:", q)
    print("A:", ans)
    print("-" * 60)

Q: How long does an authority have to respond to an information request?
A: An authority must respond to a request for information **within thirty days of receipt of the request**【RTI Act 2005, Section 7, p.11】.  
If the information sought concerns the life or liberty of a person, the response must be given **within forty‑eight hours** of receipt of the request【RTI Act 2005, Section 7, p.11】.  

This is general information, not legal advice.
------------------------------------------------------------
Q: What can I do if a product I bought is defective?
A: If a product you bought is defective, you may file a **product liability action** to claim compensation for the harm caused.

* The chapter on product liability applies to every claim for compensation for harm caused by a defective product — [Consumer Protection Act 2019, Section 82, p.31].
* You can bring a product liability action against the **product manufacturer, product service provider or product seller** — [Consumer Protectio

In [48]:
import json, numpy as np, os
json.dump(eval_set, open("/kaggle/working/eval_final.json", "w"), indent=1)
json.dump(chunks, open("/kaggle/working/chunks.json", "w"))
np.save("/kaggle/working/vecs.npy", vecs)
print(os.listdir("/kaggle/working"))

['chunks.json', 'vecs.npy', 'eval_final.json', 'results.csv', '.virtual_documents', 'eval_raw.json']


In [49]:
import pickle
os.makedirs("/kaggle/working/models", exist_ok=True)
emb.save("/kaggle/working/models/bge-small")
reranker.save("/kaggle/working/models/reranker")
pickle.dump(bm25, open("/kaggle/working/bm25.pkl", "wb"))
print(os.listdir("/kaggle/working"))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['chunks.json', 'vecs.npy', 'models', 'eval_final.json', 'bm25.pkl', 'results.csv', '.virtual_documents', 'eval_raw.json']
